# Lecture 06 — Pretraining and decoding in practice

**What evidence shows that a training run is working?**

Six ungraded exercises, in slide order. Run top to bottom. Make a prediction before each code check; use the supplied model and utilities to inspect the behavior. All students have the same expectations.

Setup from the course repository: `uv sync --extra tokenization`, then `uv run --extra tokenization python scripts/slides.py serve`. Open the lecture's **Notebook** link to preserve a personal copy under `workspace/`. This notebook and its copied assets run offline after setup. No model or data download is required.

The fixed vocabulary treats words as toy tokens. These demonstrations are separate from the graded A2 data, interfaces, and required runs. Keep A2 submissions private.

In [ ]:
from pathlib import Path
import json
import math
import sys
import torch
from torch.nn import functional as F
import matplotlib.pyplot as plt

ASSETS = Path('assets')
if not (ASSETS / 'toy_lm.py').exists():
    ASSETS = Path('slides/lecture-06/assets')
assert (ASSETS / 'toy_lm.py').exists(), 'Open this notebook with its supplied assets folder.'
sys.path.insert(0, str(ASSETS.resolve()))
from toy_lm import DecoderLM, NOA_DOCUMENTS, VOCABULARY
from training import RunConfig, ToyRun, encode, evaluate, distribution, generate, lr_at
from resume_demo import run_comparison

torch.set_num_threads(2)
print('CPU float32 | torch', torch.__version__, '| two threads')
print(dict(enumerate(VOCABULARY)))

## E01 — Build the batch (4 minutes)

Two windows are `[0,1,2,3,4]` and `[4,5,6,7,8]`. Write the input array, next-token target array, and number of loss terms. Every target is scored. These are document-local fragments.

**Your prediction:** ...

In [ ]:
windows = torch.tensor([[0,1,2,3,4], [4,5,6,7,8]])
x = windows[:, :-1].contiguous()
y = windows[:, 1:].contiguous()
print('inputs:', x.tolist(), '\ntargets:', y.tolist())
print('shape:', tuple(x.shape), '| predicted tokens:', y.numel())
assert torch.equal(x[:, 1:], y[:, :-1])
assert y.numel() == 8

<details><summary>Check your response</summary>

Inputs `[[0,1,2,3],[4,5,6,7]]`; targets `[[1,2,3,4],[5,6,7,8]]`. There are **eight** target positions. The vocabulary size does not multiply the target count. Causal masking and target shifting serve different purposes.
</details>

## E02 — Count tokens and updates (4 minutes)

One process uses microbatch size 4, length 8, accumulation count 2, and 32 optimizer updates. How many target tokens contribute per update? How many microbatches and tokens are processed in total?

**Your calculation:** ...

In [ ]:
B, T, K, updates = 4, 8, 2, 32
print('tokens/update:', B*T*K)
print('microbatches:', updates*K, '| processed tokens:', updates*B*T*K)
assert updates*B*T*K == 2048

In [ ]:
# Verify the averaging rule independently on a linear classifier.
torch.manual_seed(2)
linear = torch.nn.Linear(3, 2, bias=False).double()
features = torch.randn(4, 3, dtype=torch.float64)
labels = torch.tensor([0,1,1,0])
F.cross_entropy(linear(features), labels).backward()
full_gradient = linear.weight.grad.clone()
linear.zero_grad(set_to_none=True)
for start in [0,2]:
    loss = F.cross_entropy(linear(features[start:start+2]), labels[start:start+2])
    (loss/2).backward()
print('max gradient difference:', (linear.weight.grad-full_gradient).abs().max().item())
torch.testing.assert_close(linear.weight.grad, full_gradient, atol=1e-12, rtol=0)

<details><summary>Check your response</summary>

64 tokens/update; 64 microbatches; 2,048 processed tokens. Equal-size microbatch means each receive weight 1/2. For unequal scored-token counts, use token weights. Clip and step once after accumulation. Processed tokens may include repeated corpus tokens.
</details>

## E03 — Learning rate and clipping (5 minutes)

Use eight updates, two warmup updates, maximum LR 0.001, and minimum ratio 0.1. Find rates at zero-based updates 0, 1, and 7. Then clip gradient (3,4) to global norm 1.

**Your calculation:** ...

The cosine progress after warmup is `(s-w+1)/(N-w)`. The final applied update reaches the minimum.

In [ ]:
rates = [lr_at(s, 8, 2, .001, .1) for s in range(8)]
print('LR at updates 0,1,7:', [rates[i] for i in [0,1,7]])
parameter = torch.nn.Parameter(torch.zeros(2))
parameter.grad = torch.tensor([3.,4.])
reported_norm = torch.nn.utils.clip_grad_norm_([parameter], 1.0)
print('reported pre-clipping norm:', reported_norm.item())
print('clipped gradient:', parameter.grad.tolist())
assert reported_norm.item() == 5
assert parameter.grad.norm().item() <= 1.000001
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(range(8), rates, marker='o')
ax.set(xlabel='Optimizer update (zero-based)', ylabel='Learning rate', title='Calculated schedule')
plt.show()

<details><summary>Check your response</summary>

Rates: 0.0005, 0.001, 0.0001. Original norm 5; ideal clipped vector (0.6,0.8). PyTorch's small numerical epsilon explains tiny deviations. This bounds the gradient norm, not directly the AdamW parameter-step norm.
</details>

### Demonstration A — Fit the two Noa documents

This reproduces Lecture 05's 4,608-parameter architecture and 200-update full-batch pre-LN fit. No dropout, clipping, schedule, or decay is used here. The ambiguous cat/friend targets give a mean-loss floor of `ln(2)/10`. There is **no held-out set** in this fitting fixture.

In [ ]:
torch.manual_seed(7)
fit_model = DecoderLM().float()
assert sum(p.numel() for p in fit_model.parameters()) == 4608
windows = torch.tensor(NOA_DOCUMENTS)
fit_x, fit_y = windows[:, :-1].contiguous(), windows[:, 1:].contiguous()
optimizer = torch.optim.AdamW(fit_model.parameters(), lr=.02, betas=(.9,.999), eps=1e-8, weight_decay=0)
fit_losses = []
for completed in range(201):
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(fit_model(fit_x).flatten(0,1), fit_y.flatten())
    fit_losses.append(loss.item())
    if completed < 200:
        loss.backward()
        optimizer.step()
print('final NLL:', fit_losses[-1], '| analytic floor:', math.log(2)/10)
fig, ax = plt.subplots(figsize=(9,3))
ax.semilogy(range(201), fit_losses, label='Measured Noa full-batch fit')
ax.axhline(math.log(2)/10, color='gray', linestyle='--', label='Analytic infimum')
ax.set(xlabel='Completed optimizer updates', ylabel='NLL (nats/token)')
ax.legend()
plt.show()

### Demonstration B — Evaluate separate documents

This **different** instructor-written corpus has 16 training and 8 development documents. Exact document strings are disjoint; vocabulary and many prefixes overlap. Split before windowing; freeze the vocabulary and split.

One uniformly sampled document per update gives variable token counts. Record actual processed tokens. Full-split evaluation is token-weighted, while the sampled per-document training losses weight documents equally in expectation. This small template study does not establish natural-language generalization.

In [ ]:
corpus = json.loads((ASSETS/'corpus.json').read_text(encoding='utf-8'))
assert not set(corpus['train']) & set(corpus['dev'])
train_docs, dev_docs = ([encode(s) for s in corpus[part]] for part in ['train','dev'])
run = ToyRun(RunConfig(updates=200, warmup=20), train_docs)
curve, updates_record = [], []
for completed in range(201):
    if completed % 10 == 0:
        curve.append((completed, evaluate(run.model,train_docs)['nll'], evaluate(run.model,dev_docs)['nll']))
    if completed < 200:
        updates_record.append(run.advance())
print('final (completed updates, train NLL, dev NLL):', curve[-1])
print('actual processed tokens:', run.tokens)
fig, axes = plt.subplots(3, 1, figsize=(10,8), sharex=True)
axes[0].plot([r[0] for r in curve], [r[1] for r in curve], label='Train full split')
axes[0].plot([r[0] for r in curve], [r[2] for r in curve], label='Dev full split')
axes[0].set(ylabel='NLL (nats/token)', title='Measured toy run; complete recorded interval')
axes[0].legend()
# LR and norm belong to the update starting after k completed updates.
axes[1].plot([r['update'] for r in updates_record], [r['lr'] for r in updates_record])
axes[1].set(ylabel='Learning rate')
axes[2].plot([r['update'] for r in updates_record], [r['grad_norm'] for r in updates_record])
axes[2].axhline(1, color='gray', linestyle='--', label='Clipping threshold')
axes[2].set(xlabel='Completed updates before next update / evaluation', ylabel='Pre-clip L2 norm')
axes[2].legend()
fig.tight_layout()
plt.show()

In [ ]:
# Invented evaluation chunks: sum token losses before dividing.
chunk_means, chunk_counts = [1.,3.], [8,2]
weighted_nll = sum(a*b for a,b in zip(chunk_means,chunk_counts))/sum(chunk_counts)
print('NLL:', weighted_nll, '| perplexity:', math.exp(weighted_nll))
assert weighted_nll == 1.4

## E04 — Check the restart (6 minutes)

Predict which fields must agree for 40 continuous updates versus 20 + save + **fresh-process** restart + 20. What changes when weights, future batches, and LR are restored but AdamW's moments are reset?

**Your prediction:** ...

The supplied runner launches four CPU subprocesses and compares them. Temporary checkpoints are automatically removed. To retain a run, call `run_comparison(Path('outputs/my-new-run'))` with a new ignored output directory. See `assets/README.md` for measured runtime and environment; your machine may differ.

In [ ]:
restart = run_comparison()
print(json.dumps(restart['comparisons'], indent=2))
print('wall seconds, including worker startups:', restart['wall_seconds'])
assert restart['comparisons']['resumed']['within_tolerance']
assert restart['comparisons']['resumed']['same_batches']
assert restart['comparisons']['resumed']['same_lrs']

<details><summary>Check your response</summary>

Restore parameters, optimizer history, completed updates/schedule position, configuration, data/tokenizer identity, and RNG/sampler states before the next draw. The full restart matches at absolute tolerance 1e-7 and relative tolerance 0 in the recorded CPU environment.

The weights-only control has the same first forward loss because its weights still agree. Its next **optimizer update** differs, so the second forward loss and final parameters reveal the lost history. The continuous and control runs retain the same future batch IDs and LRs. This is a same-environment check, not a cross-device guarantee.
</details>

## E05 — Diagnose the evidence (5 minutes)

These are **schematic** records, invented for diagnosis. For each, give one plausible explanation and one useful next check.

- A: training loss falls while a fixed development loss rises.
- B: a gradient-norm spike occurs before a nonfinite loss.
- C: a very low development loss accompanies overlapping document IDs.

**Your hypotheses and checks:** ...

In [ ]:
steps = [0,1,2,3,4]
fig, axes = plt.subplots(1,2,figsize=(11,3))
axes[0].plot(steps,[2,1.4,1,.7,.5],label='Train')
axes[0].plot(steps,[2,1.6,1.7,2,2.4],label='Dev')
axes[0].set(title='A: schematic losses',xlabel='Recorded checkpoint',ylabel='NLL (nats/token)')
axes[0].legend()
axes[1].plot(steps,[1,1.2,2,35,float('nan')],marker='o')
axes[1].annotate('Next loss is nonfinite',xy=(3,35),xytext=(1,23),arrowprops={'arrowstyle':'->'})
axes[1].set(title='B: schematic gradient norms',xlabel='Recorded update',ylabel='Pre-clipping L2 norm')
fig.tight_layout()
plt.show()
train_ids, dev_ids = {'doc01','doc02','doc03'}, {'doc03','doc04'}
print('C: overlapping document IDs:', sorted(train_ids & dev_ids))

<details><summary>Discuss possible responses</summary>

A: overfitting or a split mismatch; repeat the fixed evaluation and inspect examples/checkpoints. B: optimization or numerical instability; inspect the first bad batch, LR, loss, and gradients before a controlled change. C: suspected leakage; reconstruct a disjoint split and evaluate again.

A curve alone rarely identifies a unique cause. A lower LR cannot repair leaked targets. Keep a final test split for a fixed experiment instead of using it repeatedly to choose settings.
</details>

## E06 — Compute the nucleus (5 minutes)

For probabilities `(0.4,0.3,0.2,0.1)` and top-p 0.6, identify the retained tokens and their renormalized probabilities. Which token does greedy choose? Temperature has already been applied to these probabilities.

**Your calculation:** ...

In [ ]:
logits = torch.tensor([.4,.3,.2,.1], dtype=torch.float64).log()
filtered = distribution(logits, temperature=1, top_p=.6)
print('filtered:', filtered.tolist(), '| greedy ID:', int(logits.argmax()))
torch.testing.assert_close(filtered, torch.tensor([4/7,3/7,0,0], dtype=torch.float64))
print('p=1:', distribution(logits, top_p=1).tolist())
print('tied logits, p=.5:', distribution(torch.zeros(4), top_p=.5).tolist())

<details><summary>Check your response</summary>

Keep A and B, including the token that crosses the threshold. Their original mass is 0.7, so the result is `(4/7,3/7,0,0)`. Greedy picks A. At p=1 keep every token. At ties, use smaller token IDs first. Temperature is positive; greedy is a separate mode.
</details>

### Decode from one fitted checkpoint

Use the **two-document fitting model** from Demonstration A. Keep the prompt, checkpoint, seed, and token budget fixed. The larger development experiment above is a separate model. The very short Noa completions offer limited evidence about long-form quality.

In [ ]:
prompt = [0,1,2,3,4,5,6,7,8]
print('Prefix:', ' '.join(VOCABULARY[i] for i in prompt))
for mode,tau,p in [('greedy',1,1),('sample',.7,.9),('sample',1,1)]:
    result = generate(fit_model,prompt,mode=mode,temperature=tau,top_p=p,budget=12,seed=29)
    print(mode,tau,p,'→',' '.join(VOCABULARY[i] for i in result['ids']), '| stop:', result['stop'])
assert generate(fit_model,prompt,budget=0) == {'ids':[], 'stop':'length'}

### Optional — Check the scalar AdamW appendix

First update from w=1, g=0.5, LR=0.1, decay=0.01, with zero initial moments. Set epsilon=0 only for this nonzero-gradient hand calculation: expected w=0.899. The measured model runs use epsilon=1e-8.

In [ ]:
w = torch.nn.Parameter(torch.tensor(1.,dtype=torch.float64))
scalar_optimizer = torch.optim.AdamW([w],lr=.1,weight_decay=.01,eps=0)
w.grad = torch.tensor(.5,dtype=torch.float64)
scalar_optimizer.step()
print(w.item())
assert abs(w.item()-.899) < 1e-12

## Record your evidence

Identify your batch, model/config, processed-token count, train/dev scoring convention, checkpoint comparison, and decoding settings. These are ungraded classroom activities; no additional submission is required.

- [AdamW paper](https://arxiv.org/abs/1711.05101), §2.
- [Nucleus sampling paper](https://arxiv.org/abs/1904.09751), decoding methods.
- [PyTorch checkpoint tutorial](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html#saving-loading-a-general-checkpoint-for-inference-and-or-resuming-training).
- Local `assets/README.md` identifies commands, source hashes, model/data settings, and measured timings.

A2 has its own public handout and exact protocol. Submit A2 privately through eLearning by **October 31, 2026, 23:59 Asia/Shanghai**. Project proposals are due **October 21, 23:59**.